# Timeline Mechanics EDA — Workstream 4

Owner: Anas

## Questions
1. Does fixing Arps b=0 improve temporal stability and historical backtest accuracy?
2. At which forecast quarter do Separate and Constant begin to fail?
3. What water-rate assumptions produce the largest 2028 cumulative forecasts?

## Interpretation rules
- A 2028 forecast is a conditional extrapolation, not a validated prediction.
- Historical backtests must fit only data at or before their cutoff.
- Free-b Arps is tested without b bounds; invalid future domains are reported, not clipped.
- Water caps are assumption-sensitivity experiments, not established reservoir limits.
- Aquifer Pore Volume is a relative parameter, not a documented volume in STB.

In [1]:
%pip install numpy pandas pyarrow scipy matplotlib seaborn -q


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /Users/tengkuanas/Projects/mwap-exdw-surrogate-model/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import sys, importlib, warnings, hashlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import least_squares
from scipy.integrate import cumulative_trapezoid
from scipy.special import expit, logit
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Paths ──
_curr = Path.cwd().resolve()
REPO_ROOT = None
temp = _curr
while temp != temp.parent:
    if (temp / "Anas").exists() and (temp / "data").exists():
        REPO_ROOT = temp
        break
    temp = temp.parent

if REPO_ROOT is not None:
    ROOT = REPO_ROOT / "Anas"
    SHARED = REPO_ROOT / "data" / "shared"
else:
    ROOT = Path("/Workspace/Users/tengkuanas.zainalabi@petronas.com.my/mwap-exdw-surrogate-model/Anas-db")
    SHARED = Path("/Workspace/Users/tengkuanas.zainalabi@petronas.com.my/MWAP/data")
DATA = ROOT / "data"
FIG  = ROOT / "outputs" / "figures"
TABLE = ROOT / "outputs" / "tables"
for d in [FIG, TABLE]: d.mkdir(parents=True, exist_ok=True)

# ── Import temporal_utils ──
sys.path.insert(0, str(ROOT))
import temporal_utils as tu
importlib.reload(tu)

tu_hash = hashlib.md5((ROOT / "temporal_utils.py").read_bytes()).hexdigest()[:8]
print(f"temporal_utils.py hash: {tu_hash}")

# ── Load data ──
unc, curves = tu.load_shared(SHARED)
unc_dev = unc.loc[1:85]
targets = pd.read_parquet(DATA / "temporal_targets.parquet")
fits_trials = pd.read_parquet(DATA / "temporal_fit_trials.parquet")
bt = pd.read_parquet(DATA / "crossed_backtest_predictions.parquet")
sc = pd.read_parquet(DATA / "conditional_2028_scenarios.parquet")

print(f"Backtest shape: {bt.shape}, columns: {list(bt.columns)}")
print(f"Scenario shape: {sc.shape}, columns: {list(sc.columns)}")
print(f"Available data files: {sorted(p.name for p in DATA.glob('*.parquet'))}")

# ── Constants ──
SEED = 42
YEAR_DAYS = 365.25
CUTOFF = pd.Timestamp("2008-01-01")
PARAMS = tu.PARAMS
PHASES = tu.CUM_METRICS

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "pdf.fonttype": 42, "font.size": 10,
})

def save_table(frame, name):
    frame.to_csv(TABLE / f"{name}.csv", index=False)
    return frame

def show_save(fig, name):
    fig.savefig(FIG / f"{name}.png", bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    display(fig)
    plt.close(fig)

print("\n✅ Data loaded. Ready for timeline mechanics.")

temporal_utils.py hash: 3ba3b98c


Production rows: 5725536
Excluded invalid date/value rows: 5637420


Backtest shape: (114240, 20), columns: ['case_num', 'split', 'cutoff', 'horizon_years', 'date', 'lead_years', 'metric', 'truth', 'true_anchor', 'truth_increment', 'prediction', 'predicted_rate', 'predicted_anchor', 'predicted_increment', 'cumulative_error', 'increment_error', 'anchor_error', 'method', 'mode', 'evaluation']
Scenario shape: (4800, 9), columns: ['case_num', 'date', 'method', 'oil_cum', 'gas_cum', 'water_cum', 'oil_rate', 'gas_rate', 'water_rate']
Available data files: ['case_folds.parquet', 'conditional_2028_scenarios.parquet', 'crossed_backtest_predictions.parquet', 'crossed_parameter_prediction_errors.parquet', 'cutoff_history.parquet', 'forecast_truth.parquet', 'temporal_fit_trials.parquet', 'temporal_fits.parquet', 'temporal_targets.parquet', 'uncertainty_params.parquet']

✅ Data loaded. Ready for timeline mechanics.


In [3]:
# ============================================================
# ANALYSIS 1: Matched Ablation — Free-b Arps vs Strict Exponential
# ============================================================

def arps_log_rate(t, log_q, log_D, b):
    """Arps decline in log-rate space. b=0 → exponential limit."""
    t = np.asarray(t, float)
    D = np.exp(log_D)
    if abs(b) < 1e-7:
        return log_q - D * t
    base = 1.0 + b * D * t
    result = np.full(t.shape, np.nan)
    valid = base > 0
    result[valid] = log_q - np.log1p(b * D * t[valid]) / b
    return result

def fit_decline(case, cutoff, free_b, window_years=3, n_starts=12, seed=42):
    """Fit oil decline to historical rates. free_b=True: unconstrained; False: b=0."""
    cutoff = pd.Timestamp(cutoff)
    history, _ = tu.prefix_history(curves, case, cutoff)
    h = tu.window_frame(history, f"{window_years}y")
    
    t = (h.index - h.index[0]).total_seconds().to_numpy() / (86400 * YEAR_DAYS)
    rates = h["oil_rate"].to_numpy(float)
    valid = np.isfinite(rates) & (rates > 0)
    t, rates = t[valid], rates[valid]
    if len(rates) < 6:
        raise ValueError(f"Too few positive oil rates: Case {case}")
    y = np.log(rates)
    rng = np.random.default_rng(seed + int(case))

    def residual(p):
        b_val = p[2] if free_b else 0.0
        return arps_log_rate(t, p[0], p[1], b_val) - y

    results = []
    for k in range(n_starts):
        D0 = 0.15 * np.exp(rng.normal(0, 0.5))
        b0 = 0.0 if k == 0 else rng.uniform(-0.5, 1.5)
        initial = [y[0] + rng.normal(0, 0.03), np.log(D0)]
        lower = [np.log(1e-4), np.log(1e-8)]
        upper = [np.log(max(rates.max() * 100, 1.0)), np.log(10.0)]
        if free_b:
            initial += [b0]
            lower += [-np.inf]
            upper += [np.inf]
        result = least_squares(residual, initial, bounds=(lower, upper),
                               max_nfev=5000, xtol=1e-11, ftol=1e-11, gtol=1e-11)
        if result.success and np.isfinite(result.fun).all():
            results.append(result)
    if not results:
        raise RuntimeError(f"All fits failed: Case {case}")
    best = min(results, key=lambda r: np.sum(r.fun ** 2))
    p = best.x
    b = float(p[2]) if free_b else 0.0
    return {
        "case_num": int(case), "cutoff": cutoff, "start": h.index[0],
        "log_q": float(p[0]), "log_D": float(p[1]), "b": b,
        "q_start": float(np.exp(p[0])), "D": float(np.exp(p[1])),
        "log_rate_RMSE": float(np.sqrt(np.mean(best.fun ** 2))),
        "jac_condition": float(np.linalg.cond(best.jac)),
        "dates": h.index[valid], "rates": rates, "result": best,
    }

def decline_forecast(record, dates, anchor):
    """Project decline curve forward from cutoff using fitted parameters."""
    dates = pd.DatetimeIndex(dates)
    cutoff = record["cutoff"]
    dense = pd.date_range(cutoff, dates.max(), freq="D").union(dates).sort_values()
    t = (dense - record["start"]).total_seconds().to_numpy() / (86400 * YEAR_DAYS)
    log_rate = arps_log_rate(t, record["log_q"], record["log_D"], record["b"])
    rate = np.full(len(dense), np.nan)
    valid = np.isfinite(log_rate) & (log_rate < 700)
    rate[valid] = np.exp(log_rate[valid])
    elapsed = (dense - cutoff).total_seconds().to_numpy() / 86400
    cumulative = anchor + cumulative_trapezoid(rate, elapsed, initial=0)
    frame = pd.DataFrame({"rate": rate, "cumulative": cumulative}, index=dense)
    return frame.loc[dates]

# Select 12 training cases spanning porosity range
eligible = unc_dev.loc[1:70].sort_values("Porosity Multiplier")
positions = np.unique(np.round(np.linspace(0, len(eligible) - 1, 12)).astype(int))
selected_cases = eligible.iloc[positions].index.to_numpy(int)
future_dates = pd.date_range("2008-04-01", "2028-01-01", freq="QS")

print(f"Selected 12 cases: {selected_cases}")
print(f"Porosity range: {unc_dev.loc[selected_cases, 'Porosity Multiplier'].min():.3f} → {unc_dev.loc[selected_cases, 'Porosity Multiplier'].max():.3f}")

fits_dict = {}
forecasts_dict = {}
fit_rows = []

for case in selected_cases:
    anchor = tu.interpolate_truth(curves[(case, "oil_cum")], [CUTOFF])[0]
    for label, free_b in [("Free-b Arps", True), ("Strict exponential", False)]:
        record = fit_decline(case, CUTOFF, free_b)
        fits_dict[(case, label)] = record
        forecasts_dict[(case, label)] = decline_forecast(record, future_dates, anchor)
        fit_rows.append({
            "case_num": case,
            "Porosity Multiplier": unc_dev.loc[case, "Porosity Multiplier"],
            "fit": label,
            "q_start": record["q_start"], "D": record["D"], "b": record["b"],
            "log_rate_RMSE": record["log_rate_RMSE"],
            "jac_condition": record["jac_condition"],
        })

fit_table = pd.DataFrame(fit_rows)
save_table(fit_table, "timeline_matched_decline_fits")
display(fit_table)

Selected 12 cases: [53 66 36 70  8 61 52 32 69 40 35 12]
Porosity range: 0.801 → 1.494


,case_num,Porosity Multiplier,fit,q_start,D,b,log_rate_RMSE,jac_condition
0,53,0.801219,Free-b Arps,6027.699777,0.602816,2.902443,0.015044,66.619510
1,53,0.801219,Strict exponential,5409.806201,0.194067,0.000000,0.050768,5.991479
2,66,0.876612,Free-b Arps,6382.837291,0.541249,2.893205,0.013272,72.415371
3,66,0.876612,Strict exponential,5789.598967,0.186125,0.000000,0.046620,6.205988
4,36,0.903606,Free-b Arps,6681.794311,0.506685,3.034478,0.011109,80.175338
5,36,0.903606,Strict exponential,6099.970888,0.176281,0.000000,0.042848,6.501454
6,70,0.986349,Free-b Arps,7238.280733,0.428710,2.928164,0.009109,89.667974
7,70,0.986349,Strict exponential,6704.552569,0.167405,0.000000,0.036830,6.800399
8,8,1.027024,Free-b Arps,7588.196390,0.342609,2.397573,0.005687,93.749369
9,8,1.027024,Strict exponential,7164.650434,0.166408,0.000000,0.027712,6.836117


In [4]:
# Overlay free-b vs exponential forecasts + endpoint comparison
fig, axes = plt.subplots(4, 3, figsize=(16, 15))
endpoint_rows = []

for ax, case in zip(axes.flat, selected_cases):
    anchor = tu.interpolate_truth(curves[(case, "oil_cum")], [CUTOFF])[0]
    hist_dates = fits_dict[(case, "Free-b Arps")]["dates"]
    hist_cum = tu.interpolate_truth(curves[(case, "oil_cum")], hist_dates)
    ax.plot(hist_dates, hist_cum / 1e6, color="black", lw=1.5, label="Historical")

    for label, color in [("Free-b Arps", "#D55E00"), ("Strict exponential", "#0072B2")]:
        fc = forecasts_dict[(case, label)]
        ax.plot(future_dates, fc["cumulative"] / 1e6, color=color, label=label)
        endpoint_rows.append({
            "case_num": case, "fit": label,
            "oil_cum_2028": fc["cumulative"].iloc[-1],
            "oil_rate_2028": fc["rate"].iloc[-1],
            "invalid_points": int((~np.isfinite(fc.to_numpy())).any(axis=1).sum()),
            "b": fits_dict[(case, label)]["b"],
        })
    ax.axvline(CUTOFF, color="grey", ls="--", lw=1)
    ax.set(title=f"Case {case} | por={unc_dev.loc[case, 'Porosity Multiplier']:.3f}",
           ylabel="Oil cum (M STB)")

axes.flat[0].legend(fontsize=8)
fig.suptitle("Matched oil decline fits: identical history, different b constraint", y=1.01)
fig.tight_layout()
show_save(fig, "timeline_free_b_vs_exponential_2028")

# Endpoint comparison table
endpoint_table = pd.DataFrame(endpoint_rows)
comparison = endpoint_table.pivot(index="case_num", columns="fit", values="oil_cum_2028").reset_index()
comparison["endpoint_diff_pct"] = 100 * (comparison["Free-b Arps"] / comparison["Strict exponential"] - 1)
save_table(endpoint_table, "timeline_decline_endpoints")
save_table(comparison, "timeline_decline_endpoint_comparison")
display(comparison)

print(f"\nMean endpoint difference: {comparison['endpoint_diff_pct'].mean():.2f}%")
print(f"Max endpoint difference: {comparison['endpoint_diff_pct'].abs().max():.2f}%")

<Figure size 1920x1800 with 12 Axes>

fit,case_num,Free-b Arps,Strict exponential,endpoint_diff_pct
0,8,5.606050e+07,4.412664e+07,27.044587
1,12,7.438887e+07,5.611377e+07,32.567946
2,32,6.809320e+07,5.217021e+07,30.521233
3,35,7.603677e+07,5.738202e+07,32.509747
4,36,5.126631e+07,3.968082e+07,29.196692
5,40,7.245902e+07,5.517384e+07,31.328595
6,52,6.262559e+07,4.886156e+07,28.169440
7,53,4.614973e+07,3.612179e+07,27.761487
8,61,5.784560e+07,4.555307e+07,26.985090
9,66,4.861973e+07,3.789645e+07,28.296281



Mean endpoint difference: 29.50%
Max endpoint difference: 32.57%


In [5]:
# Rolling-origin backtests: fit at 2003/2004/2005, forecast to their horizons
EXPERIMENTS = [
    ("2003-01-01", 3), ("2003-01-01", 5),
    ("2004-01-01", 3), ("2005-01-01", 3),
]

ablation_rows = []

for origin_text, horizon in EXPERIMENTS:
    origin = pd.Timestamp(origin_text)
    dates = pd.date_range(origin + pd.DateOffset(months=3),
                          origin + pd.DateOffset(years=horizon), freq="QS")
    for case in selected_cases:
        anchor = tu.interpolate_truth(curves[(case, "oil_cum")], [origin])[0]
        truth = tu.interpolate_truth(curves[(case, "oil_cum")], dates)
        true_increment = truth - anchor

        for label, free_b in [("Free-b Arps", True), ("Strict exponential", False)]:
            try:
                record = fit_decline(case, origin, free_b)
                fc = decline_forecast(record, dates, anchor)
                pred_increment = fc["cumulative"].to_numpy() - anchor
                finite = np.isfinite(pred_increment).all()
                scale = np.sqrt(np.mean(true_increment ** 2))
                score = (np.sqrt(np.mean((pred_increment - true_increment) ** 2)) / scale
                         if finite and scale > 0 else np.nan)
            except Exception:
                score = np.nan
                finite = False
            ablation_rows.append({
                "case_num": case, "origin": origin,
                "horizon_years": horizon, "fit": label,
                "oil_increment_NRMSE": score,
                "complete_forecast": finite,
            })

ablation = pd.DataFrame(ablation_rows)
save_table(ablation, "timeline_matched_ablation_case_scores")

# Summary by experiment
summary = ablation.groupby(["origin", "horizon_years", "fit"], as_index=False).agg(
    mean_NRMSE=("oil_increment_NRMSE", "mean"),
    median_NRMSE=("oil_increment_NRMSE", "median"),
    complete=("complete_forecast", "sum"),
)
save_table(summary, "timeline_matched_ablation_summary")
display(summary)

# Paired comparison
paired = ablation.pivot(index=["case_num", "origin", "horizon_years"],
                        columns="fit", values="oil_increment_NRMSE").dropna().reset_index()
paired["exp_minus_free"] = paired["Strict exponential"] - paired["Free-b Arps"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=paired, x="Free-b Arps", y="Strict exponential",
                hue="origin", palette="viridis", ax=axes[0])
limit = max(paired[["Free-b Arps", "Strict exponential"]].max().max(), 0.01)
axes[0].plot([0, limit], [0, limit], "--", color="black")
axes[0].set(title="Oil increment accuracy: matched pairs")

sns.boxplot(data=paired, x="horizon_years", y="exp_minus_free",
            color="#CDE3F2", ax=axes[1])
axes[1].axhline(0, color="black", ls="--")
axes[1].set(title="Negative = strict exponential is BETTER",
            ylabel="Exponential NRMSE − free-b NRMSE")
fig.tight_layout()
show_save(fig, "timeline_matched_ablation_accuracy")

wins = (paired["exp_minus_free"] < 0).sum()
total = len(paired)
print(f"\nStrict exponential wins {wins}/{total} matched pairs ({100*wins/total:.1f}%)")

,origin,horizon_years,fit,mean_NRMSE,median_NRMSE,complete
0,2003-01-01,3,Free-b Arps,NaN,NaN,0
1,2003-01-01,3,Strict exponential,0.270510,0.290268,12
2,2003-01-01,5,Free-b Arps,NaN,NaN,0
3,2003-01-01,5,Strict exponential,0.377442,0.419028,12
4,2004-01-01,3,Free-b Arps,0.040291,0.040291,2
5,2004-01-01,3,Strict exponential,0.135527,0.123435,12
6,2005-01-01,3,Free-b Arps,0.076311,0.068539,10
7,2005-01-01,3,Strict exponential,0.041108,0.026848,12


<Figure size 1560x600 with 2 Axes>


Strict exponential wins 8/12 matched pairs (66.7%)


In [6]:
# ============================================================
# ANALYSIS 2: Error Compounding Trajectory
# ============================================================
# Use crossed_backtest_predictions.parquet (114,240 rows from notebook 02)

print(f"Backtest columns: {list(bt.columns)}")
print(f"Backtest methods: {sorted(bt['method'].unique())}")
print(f"Backtest shape: {bt.shape}")

# Identify key columns (adapt to actual schema)
bt_work = bt.copy()

# Compute lead quarter from cutoff/origin
if "origin" in bt_work.columns:
    origin_col = "origin"
elif "cutoff" in bt_work.columns:
    origin_col = "cutoff"
else:
    raise ValueError(f"Need origin/cutoff column. Available: {list(bt_work.columns)}")

bt_work[origin_col] = pd.to_datetime(bt_work[origin_col])
bt_work["date"] = pd.to_datetime(bt_work["date"])

bt_work["lead_months"] = (
    (bt_work["date"].dt.year - bt_work[origin_col].dt.year) * 12
    + bt_work["date"].dt.month - bt_work[origin_col].dt.month
)
bt_work["lead_quarter"] = bt_work["lead_months"] // 3

# Filter to known phases
phase_col = "phase" if "phase" in bt_work.columns else "metric"
bt_work = bt_work[bt_work[phase_col].isin(PHASES)].copy()

# Identify prediction and truth columns
pred_col = next(c for c in ["prediction", "pred", "predicted", "pred_cum"] if c in bt_work.columns)
truth_col = next(c for c in ["truth", "actual", "true", "true_cum"] if c in bt_work.columns)

# Get anchor at origin for each case/phase/origin
def get_anchor(row):
    case = int(row["case_num"])
    phase = str(row[phase_col])
    origin = pd.Timestamp(row[origin_col])
    try:
        return tu.interpolate_truth(curves[(case, phase)], [origin])[0]
    except Exception:
        return np.nan

print("Computing anchors...")
anchor_keys = bt_work[["case_num", phase_col, origin_col]].drop_duplicates()
anchor_keys["anchor"] = anchor_keys.apply(get_anchor, axis=1)
bt_work = bt_work.merge(anchor_keys, on=["case_num", phase_col, origin_col], how="left")

bt_work["true_increment"] = bt_work[truth_col] - bt_work["anchor"]
bt_work["total_error"] = bt_work[pred_col] - bt_work[truth_col]

# Score quarter-by-quarter
method_col = "method" if "method" in bt_work.columns else "decoder"
mode_col = "mode" if "mode" in bt_work.columns else None
split_col = "split" if "split" in bt_work.columns else "evaluation"
horizon_col = next((c for c in ["horizon_years", "horizon", "forecast_horizon_years"] if c in bt_work.columns), None)

group_cols = [method_col, origin_col, phase_col, "lead_quarter"]
if split_col in bt_work.columns: group_cols.insert(1, split_col)
if mode_col and mode_col in bt_work.columns: group_cols.insert(1, mode_col)
if horizon_col and horizon_col in bt_work.columns: group_cols.insert(2, horizon_col)

quarter_rows = []
for keys, group in bt_work.groupby(group_cols, dropna=False):
    row = dict(zip(group_cols, keys))
    normalizer = np.sqrt(np.mean(group["true_increment"].to_numpy() ** 2))
    errors = group["total_error"].to_numpy()
    valid = np.isfinite(errors).all() and normalizer > 0
    row["cases"] = len(group)
    row["increment_NRMSE"] = np.sqrt(np.mean(errors ** 2)) / normalizer if valid else np.nan
    row["cumulative_RMSE"] = np.sqrt(np.mean(errors ** 2)) if valid else np.nan
    row["bias"] = np.mean(errors) / normalizer if valid else np.nan
    quarter_rows.append(row)

quarter_scores = pd.DataFrame(quarter_rows)
save_table(quarter_scores, "timeline_quarter_resolved_scores")
print(f"Quarter scores: {quarter_scores.shape[0]} rows")
display(quarter_scores.head(12))

Backtest columns: ['case_num', 'split', 'cutoff', 'horizon_years', 'date', 'lead_years', 'metric', 'truth', 'true_anchor', 'truth_increment', 'prediction', 'predicted_rate', 'predicted_anchor', 'predicted_increment', 'cumulative_error', 'increment_error', 'anchor_error', 'method', 'mode', 'evaluation']
Backtest methods: ['Constant', 'Liquid-WC-time', 'Liquid-WC-volume', 'Separate']
Backtest shape: (114240, 20)
Computing anchors...


Quarter scores: 2688 rows


,method,mode,horizon_years,split,cutoff,metric,lead_quarter,cases,increment_NRMSE,cumulative_RMSE,bias
0,Constant,History-conditioned,3,train,2003-01-01,gas_cum,1,70,0.051375,1.939081e+04,0.048949
1,Constant,History-conditioned,3,train,2003-01-01,gas_cum,2,70,0.090345,6.612069e+04,0.087517
2,Constant,History-conditioned,3,train,2003-01-01,gas_cum,3,70,0.126555,1.351940e+05,0.123432
3,Constant,History-conditioned,3,train,2003-01-01,gas_cum,4,70,0.158935,2.206309e+05,0.155617
4,Constant,History-conditioned,3,train,2003-01-01,gas_cum,5,70,0.189213,3.197506e+05,0.185768
5,Constant,History-conditioned,3,train,2003-01-01,gas_cum,6,70,0.218907,4.328896e+05,0.215369
6,Constant,History-conditioned,3,train,2003-01-01,gas_cum,7,70,0.249016,5.613176e+05,0.245429
7,Constant,History-conditioned,3,train,2003-01-01,gas_cum,8,70,0.279325,7.031525e+05,0.275724
8,Constant,History-conditioned,3,train,2003-01-01,gas_cum,9,70,0.310009,8.558742e+05,0.306432
9,Constant,History-conditioned,3,train,2003-01-01,gas_cum,10,70,0.343165,1.026373e+06,0.339636


In [7]:
# Plot error trajectories: Separate vs Constant by experiment
colors = {"oil_cum": "#0072B2", "gas_cum": "#CC79A7", "water_cum": "#009E73"}

def method_family(name):
    text = str(name).lower().replace("_", " ").replace("-", " ")
    if "separate" in text: return "Separate"
    if "constant" in text: return "Constant"
    return None

plot_qs = quarter_scores.copy()
plot_qs["decoder"] = plot_qs[method_col].map(method_family)
plot_qs = plot_qs.dropna(subset=["decoder"])

if len(plot_qs["decoder"].unique()) < 2:
    print(f"Only found: {plot_qs['decoder'].unique()}. Plotting what's available.")

styles = {"Separate": "-", "Constant": "--"}
diag_rows = []

# Get unique experiments
if horizon_col and horizon_col in plot_qs.columns:
    experiments = plot_qs.groupby([origin_col, horizon_col]).size().reset_index()[[origin_col, horizon_col]]
else:
    experiments = plot_qs[[origin_col]].drop_duplicates()

n_exp = min(len(experiments), 4)
fig, axes = plt.subplots(2, n_exp, figsize=(5 * n_exp, 9), sharex="col", squeeze=False)

for col_idx in range(n_exp):
    exp = experiments.iloc[col_idx]
    origin = exp[origin_col]
    mask = plot_qs[origin_col] == origin
    if horizon_col and horizon_col in plot_qs.columns:
        mask &= plot_qs[horizon_col] == exp[horizon_col]
        title = f"{pd.Timestamp(origin).year}→{int(exp[horizon_col])}yr"
    else:
        title = f"{pd.Timestamp(origin).year}"
    
    experiment = plot_qs[mask]
    for (decoder, phase), line in experiment.groupby(["decoder", phase_col]):
        line = line.sort_values("lead_quarter")
        ls = styles.get(decoder, "-")
        c = colors.get(phase, "gray")
        label = f"{phase.replace('_cum', '')} — {decoder}"
        axes[0, col_idx].plot(line["lead_quarter"], line["increment_NRMSE"],
                              color=c, ls=ls, lw=2, label=label)
        axes[1, col_idx].plot(line["lead_quarter"], line["cumulative_RMSE"] / 1e6,
                              color=c, ls=ls, lw=2)
        
        vals = line["increment_NRMSE"].to_numpy()
        if len(vals) > 0 and np.isfinite(vals).any():
            diag_rows.append({
                "origin": origin, "decoder": decoder, "phase": phase,
                "first_q_NRMSE": vals[0] if np.isfinite(vals[0]) else np.nan,
                "last_q_NRMSE": vals[-1] if np.isfinite(vals[-1]) else np.nan,
                "peak_NRMSE": np.nanmax(vals),
                "peak_quarter": line["lead_quarter"].iloc[np.nanargmax(vals)],
            })
    
    axes[0, col_idx].set_title(title)
    axes[0, col_idx].set_ylabel("Increment NRMSE")
    axes[1, col_idx].set_ylabel("Cumulative RMSE (M units)")
    axes[1, col_idx].set_xlabel("Lead quarters")

axes[0, 0].legend(fontsize=7, ncol=2)
fig.suptitle("Error compounding trajectory: Separate (solid) vs Constant (dashed)", y=1.01)
fig.tight_layout()
show_save(fig, "timeline_error_trajectory")

diagnostics = pd.DataFrame(diag_rows)
save_table(diagnostics, "timeline_error_trajectory_diagnostics")
display(diagnostics)

print("\nInterpretation guide:")
print("  Immediate spike + small RMSE → small-denominator effect")
print("  Large first-quarter RMSE → anchor/cutoff continuity problem")
print("  Gradual increase → decline misspecification")
print("  Late acceleration → unstable extrapolation or water dynamics")

<Figure size 2400x1080 with 8 Axes>

,origin,decoder,phase,first_q_NRMSE,last_q_NRMSE,peak_NRMSE,peak_quarter
0,2003-01-01,Constant,gas_cum,0.051375,0.401676,0.412534,12
1,2003-01-01,Constant,oil_cum,0.052257,0.401851,0.413284,12
2,2003-01-01,Constant,water_cum,0.035859,0.187643,0.187643,12
3,2003-01-01,Separate,gas_cum,0.042399,0.262066,0.271307,12
4,2003-01-01,Separate,oil_cum,0.042663,0.261677,0.270959,12
5,2003-01-01,Separate,water_cum,0.019099,0.214069,0.243135,12
6,2003-01-01,Constant,gas_cum,0.051375,0.633218,0.649125,20
7,2003-01-01,Constant,oil_cum,0.052257,0.633508,0.650027,20
8,2003-01-01,Constant,water_cum,0.035859,0.254405,0.254405,20
9,2003-01-01,Separate,gas_cum,0.042399,0.376156,0.386218,20



Interpretation guide:
  Immediate spike + small RMSE → small-denominator effect
  Large first-quarter RMSE → anchor/cutoff continuity problem
  Gradual increase → decline misspecification
  Late acceleration → unstable extrapolation or water dynamics


In [8]:
# ============================================================
# ANALYSIS 3: Physical Water Bounds Diagnosis
# ============================================================
# Use conditional_2028_scenarios.parquet from notebook 02

print(f"Scenario columns: {list(sc.columns)}")
print(f"Scenario shape: {sc.shape}")

# Schema: wide format with water_cum, water_rate as direct columns
sc_work = sc.copy()
sc_work["date"] = pd.to_datetime(sc_work["date"])

# Filter to post-cutoff, dev cases
water = sc_work[
    (sc_work["case_num"].between(1, 85))
    & (sc_work["date"] > CUTOFF)
    & (sc_work["date"] <= pd.Timestamp("2028-01-01"))
].copy()

print(f"Water scenario rows: {len(water)}")
print(f"Methods: {sorted(water['method'].unique())}")

# Get 2028 endpoints and pick top 5 per method
endpoints = water[water["date"] == pd.Timestamp("2028-01-01")].copy()
top = endpoints.sort_values("water_cum", ascending=False).groupby("method", group_keys=False).head(5)
print(f"Top water cases: {sorted(top['case_num'].unique())}")

# Observed 2008 water rate benchmark
OBS_WATER = float(curves[(0, "water_rate")].iloc[-1]) if (0, "water_rate") in curves else 7926.36
print(f"Observed 2008 water rate benchmark: {OBS_WATER:.1f} STB/day")

# Compute interval-average water rates
water_rate_rows = []
water_diag_rows = []

for _, ep in top.iterrows():
    case = int(ep["case_num"])
    method = ep["method"]
    group = water[
        (water["method"] == method) & (water["case_num"] == case)
    ].sort_values("date")
    
    # Get anchor from history
    anchor = tu.interpolate_truth(curves[(case, "water_cum")], [CUTOFF])[0]
    
    dates = pd.DatetimeIndex([CUTOFF]).append(pd.DatetimeIndex(group["date"]))
    cumulative = np.r_[anchor, group["water_cum"].to_numpy(float)]
    days = np.diff(dates.asi8) / (86400 * 1e9)
    rates = np.diff(cumulative) / days
    midpoints = dates[:-1] + (dates[1:] - dates[:-1]) / 2
    
    total_days = (dates[-1] - CUTOFF).total_seconds() / 86400
    avg_rate = (cumulative[-1] - anchor) / total_days
    
    water_diag_rows.append({
        "method": method, "case_num": case,
        "Aquifer_PV": unc_dev.loc[case, "Aquifer Pore Volume"],
        "Porosity_Mult": unc_dev.loc[case, "Porosity Multiplier"],
        "water_anchor_2008": anchor,
        "water_cum_2028": cumulative[-1],
        "forecast_increment": cumulative[-1] - anchor,
        "avg_water_rate": avg_rate,
        "avg_to_observed_2008": avg_rate / OBS_WATER,
        "first_rate": rates[0], "last_rate": rates[-1],
        "max_rate": rates.max(),
        "negative_intervals": int((rates < -1e-6).sum()),
    })
    for d, mid, dy, r in zip(group["date"], midpoints, days, rates):
        water_rate_rows.append({
            "method": method, "case_num": case,
            "date": d, "midpoint": mid, "interval_days": dy, "water_rate": r,
        })

water_rate_df = pd.DataFrame(water_rate_rows)
water_diag = pd.DataFrame(water_diag_rows)
save_table(water_rate_df, "timeline_high_water_rates")
save_table(water_diag, "timeline_high_water_diagnosis")
display(water_diag)

# Plot rate trajectories
for method, subset in water_rate_df.groupby("method"):
    fig, ax = plt.subplots(figsize=(11, 6))
    for case, line in subset.groupby("case_num"):
        ax.plot(line["midpoint"], line["water_rate"], lw=2, label=f"Case {case}")
    ax.axhline(OBS_WATER, color="black", ls="--", label="Observed 2008 benchmark")
    ax.set(title=f"High-water rate trajectories | {method}",
           xlabel="Date", ylabel="Interval-average water rate (STB/day)")
    ax.legend()
    fig.tight_layout()
    show_save(fig, f"timeline_high_water_rates_{method.replace(' ', '_')}")

print("\nKey question: are these rates increasing, plateauing, or declining?")
print(f"Max avg-to-observed ratio: {water_diag['avg_to_observed_2008'].max():.2f}x")

Scenario columns: ['case_num', 'date', 'method', 'oil_cum', 'gas_cum', 'water_cum', 'oil_rate', 'gas_rate', 'water_rate']
Scenario shape: (4800, 9)
Water scenario rows: 4800
Methods: ['Constant', 'Liquid-WC-time', 'Liquid-WC-volume', 'Separate']
Top water cases: [np.int64(75), np.int64(80), np.int64(82), np.int64(83), np.int64(84), np.int64(85)]
Observed 2008 water rate benchmark: 7926.4 STB/day


,method,case_num,Aquifer_PV,Porosity_Mult,water_anchor_2008,water_cum_2028,forecast_increment,avg_water_rate,avg_to_observed_2008,first_rate,last_rate,max_rate,negative_intervals
0,Separate,80,101.513718,0.916677,17179406.0,1.147160e+08,9.753660e+07,13352.033414,1.684510,8.657727e+06,1.681381e+07,1.681381e+07,0
1,Separate,84,78.411512,0.884133,17457360.0,1.134803e+08,9.602291e+07,13144.819372,1.658367,8.265110e+06,1.650258e+07,1.650258e+07,0
2,Separate,85,128.541520,0.836516,18594626.0,1.128863e+08,9.429167e+07,12907.826238,1.628468,8.480044e+06,1.539385e+07,1.539385e+07,0
3,Separate,83,117.838069,0.946182,16879450.0,1.126779e+08,9.579845e+07,13114.093585,1.654491,8.948684e+06,1.653944e+07,1.653944e+07,0
4,Separate,82,169.708853,1.453704,11476542.0,1.009247e+08,8.944813e+07,12244.781170,1.544817,8.346538e+06,1.661632e+07,1.661632e+07,0
5,Liquid-WC-time,85,128.541520,0.836516,18594626.0,9.027883e+07,7.168420e+07,9813.032058,1.238025,8.441238e+06,9.015507e+06,1.029974e+07,0
6,Liquid-WC-volume,85,128.541520,0.836516,18594626.0,8.986113e+07,7.126650e+07,9755.852349,1.230811,8.440926e+06,8.973643e+06,1.022390e+07,0
7,Liquid-WC-time,84,78.411512,0.884133,17457360.0,8.633656e+07,6.887920e+07,9429.047954,1.189581,8.253686e+06,8.631572e+06,9.916752e+06,0
8,Liquid-WC-time,80,101.513718,0.916677,17179406.0,8.590560e+07,6.872620e+07,9408.103692,1.186938,8.646246e+06,8.647783e+06,9.874810e+06,0
9,Liquid-WC-volume,84,78.411512,0.884133,17457360.0,8.585298e+07,6.839562e+07,9362.849899,1.181229,8.253197e+06,8.582522e+06,9.828635e+06,0


<Figure size 1320x720 with 1 Axes>

<Figure size 1320x720 with 1 Axes>

<Figure size 1320x720 with 1 Axes>

<Figure size 1320x720 with 1 Axes>


Key question: are these rates increasing, plateauing, or declining?
Max avg-to-observed ratio: 1.68x


In [9]:
# Water-cap sensitivity: what happens if we limit water rates?
COMPRESSION_SCALE = OBS_WATER

cap_rows = []
cap_curve_rows = []

for (method, case), group in water_rate_df.groupby(["method", "case_num"]):
    group = group.sort_values("date")
    anchor = tu.interpolate_truth(curves[(int(case), "water_cum")], [CUTOFF])[0]
    original_rate = group["water_rate"].to_numpy(float)
    durations = group["interval_days"].to_numpy(float)
    rate = np.maximum(original_rate, 0.0)
    
    variants = {"Original": original_rate}
    for mult in [2.0, 3.0]:
        cap = mult * OBS_WATER
        variants[f"Hard cap {mult:.0f}x"] = np.minimum(rate, cap)
        # log1p compression above benchmark
        excess = np.maximum(rate - OBS_WATER, 0)
        compressed = np.where(
            rate <= OBS_WATER, rate,
            OBS_WATER + COMPRESSION_SCALE * np.log1p(excess / COMPRESSION_SCALE)
        )
        variants[f"log1p + cap {mult:.0f}x"] = np.minimum(compressed, cap)
    
    original_endpoint = anchor + np.sum(original_rate * durations)
    for label, vrate in variants.items():
        cumulative = anchor + np.cumsum(vrate * durations)
        cap_rows.append({
            "method": method, "case_num": case, "variant": label,
            "water_cum_2028": cumulative[-1],
            "reduction_pct": 100 * (1 - cumulative[-1] / original_endpoint) if original_endpoint > 0 else 0,
            "max_rate": vrate.max(), "avg_rate": np.sum(vrate * durations) / durations.sum(),
            "changed_fraction": np.mean(~np.isclose(vrate, original_rate)),
        })

cap_results = pd.DataFrame(cap_rows)
save_table(cap_results, "timeline_water_cap_sensitivity")
display(cap_results)

# Plot cap sensitivity per case
for method in water_rate_df["method"].unique():
    cases = sorted(water_rate_df[water_rate_df["method"] == method]["case_num"].unique())
    fig, axes = plt.subplots(len(cases), 2, figsize=(14, 3.2 * len(cases)), squeeze=False)
    for row, case in enumerate(cases):
        grp = water_rate_df[(water_rate_df["method"] == method) & (water_rate_df["case_num"] == case)].sort_values("date")
        anchor = tu.interpolate_truth(curves[(int(case), "water_cum")], [CUTOFF])[0]
        original_rate = grp["water_rate"].to_numpy(float)
        durations = grp["interval_days"].to_numpy(float)
        rate = np.maximum(original_rate, 0.0)
        
        for mult in [2.0, 3.0]:
            cap = mult * OBS_WATER
            capped = np.minimum(rate, cap)
            cum_orig = anchor + np.cumsum(original_rate * durations)
            cum_cap = anchor + np.cumsum(capped * durations)
            
            axes[row, 0].plot(grp["date"], original_rate, lw=2, label="Original")
            axes[row, 0].plot(grp["date"], capped, ls="--", lw=1.5, label=f"Cap {mult:.0f}x")
            axes[row, 0].axhline(OBS_WATER, color="grey", ls=":", lw=1)
            
            axes[row, 1].plot(grp["date"], cum_orig / 1e6, lw=2, label="Original")
            axes[row, 1].plot(grp["date"], cum_cap / 1e6, ls="--", lw=1.5, label=f"Cap {mult:.0f}x")
        
        axes[row, 0].set(title=f"Case {case}: water rate", ylabel="STB/day")
        axes[row, 1].set(title=f"Case {case}: water cumulative", ylabel="M STB")
    
    axes[0, 1].legend(fontsize=8)
    fig.suptitle(f"Water cap sensitivity | {method}", y=1.01)
    fig.tight_layout()
    show_save(fig, f"timeline_water_cap_{method.replace(' ', '_')}")

print("\n⚠️ These caps are stress tests, not inferred engineering limits.")
print("Do not select a cap just because it makes endpoints look reasonable.")

,method,case_num,variant,water_cum_2028,reduction_pct,max_rate,avg_rate,changed_fraction
0,Constant,75,Original,7.325025e+07,0.000000,7.931987e+06,7.923703e+06,0.0
1,Constant,75,Hard cap 2x,1.548341e+07,78.862316,1.585272e+04,1.585272e+04,1.0
2,Constant,75,log1p + cap 2x,1.548341e+07,78.862316,1.585272e+04,1.585272e+04,1.0
3,Constant,75,Hard cap 3x,1.554131e+07,78.783269,2.377909e+04,2.377909e+04,1.0
4,Constant,75,log1p + cap 3x,1.554131e+07,78.783269,2.377909e+04,2.377909e+04,1.0
...,...,...,...,...,...,...,...,...
95,Separate,85,Original,1.128863e+08,0.000000,1.539385e+07,1.290783e+07,0.0
96,Separate,85,Hard cap 2x,1.871043e+07,83.425419,1.585272e+04,1.585272e+04,1.0
97,Separate,85,log1p + cap 2x,1.871043e+07,83.425419,1.585272e+04,1.585272e+04,1.0
98,Separate,85,Hard cap 3x,1.876833e+07,83.374127,2.377909e+04,2.377909e+04,1.0


<Figure size 1680x1920 with 10 Axes>

<Figure size 1680x1920 with 10 Axes>

<Figure size 1680x1920 with 10 Axes>

<Figure size 1680x1920 with 10 Axes>


⚠️ These caps are stress tests, not inferred engineering limits.
Do not select a cap just because it makes endpoints look reasonable.
